In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf, tensorflow.keras as keras

In [ ]:
df_raw = pd.read_csv('XY_radon.csv'); df_raw

In [ ]:
from sklearn.model_selection import train_test_split,KFold, StratifiedGroupKFold,StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures,FunctionTransformer,LabelEncoder,OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.pipeline import make_pipeline, Pipeline, clone
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import mean_squared_error
from pyproj import Transformer

In [ ]:
df = df_raw
df_train = df[df.isnull()['Y']==False]
df_train = df_train.drop_duplicates()
df_test = df.query('Y!=Y').drop('Y', axis=1)

In [ ]:
X = df_train.drop(columns=['Y'])
y = df_train['Y']
print(f'Length train data:{X.shape[0]}, Length test data:{df_test.shape[0]}')

In [ ]:
#combined feature engineering
class RadonFeatureEngineering(BaseEstimator, TransformerMixin):
    """
    Advanced Feature Engineering inspired by past winning solutions (SP25 rank 1 & 2, FA24 rank 1 & 2):
    1. US Albers projected coordinates (km).
    2. Log & inverse transformations on skewed features (pcterr, Uppm, adjwt) reflecting
       detector Poisson counting error physics.
    3. Structural & spatial interactions: floor * Uppm, x * Uppm, y * Uppm.
    4. Basement indicator based on floor == 0 or basement == 'Y'.
    5. Categorical state group identifier for dense entity embedding.
    """
    def __init__(self, lat_col="lat", lon_col="lon"):
        self.lat_col = lat_col
        self.lon_col = lon_col

    def fit(self, X, y=None):
        self.transformer_ = Transformer.from_crs("EPSG:4326", "EPSG:5070", always_xy=True)
        return self

    def transform(self, X):
        df = X.copy()
        x, y = self.transformer_.transform(df[self.lon_col].to_numpy(), df[self.lat_col].to_numpy())
        df['x'] = x / 1000.0
        df['y'] = y / 1000.0
        df['pcterr_zero'] = (df['pcterr'] <= 0).astype(float)
        df['log_pcterr'] = np.log(np.maximum(df['pcterr'], 0.1))
        df['inv_pcterr'] = 1.0 / (df['pcterr'] + 1.0)
        df['log_Uppm'] = np.log1p(np.maximum(df['Uppm'], 0.0))
        df['log_adjwt'] = np.log1p(np.maximum(df['adjwt'], 0.0))
        df['floor_x_log_Uppm'] = df['floor'] * df['log_Uppm']
        df['x_x_log_Uppm'] = (df['x'] / 1000.0) * df['log_Uppm']
        df['y_x_log_Uppm'] = (df['y'] / 1000.0) * df['log_Uppm']
        df['is_basement'] = ((df['floor'] == 0) | (df['basement'] == 'Y')).astype(float)
        df['grp_id'] = df['state2'].astype(str)
        return df

def build_improved_NN_model(input_dim1, num_groups, embedding_dim=8):
    """
    Improved Deep Neural Network Architecture (SP25 rank 1 & FA24 rank 3):
    - Entity embedding layer for categorical state groups.
    - Input batch normalization for feature scale alignment.
    - 3 hidden dense layers with ReLU activations and Glorot uniform initialization.
    - Batch normalization and dropout (0.2, 0.1) to prevent overfitting.
    - Cosine decay learning rate scheduler for smooth convergence.
    - Softplus output layer predicting log1p(y) ensuring non-negativity.
    """
    # tf.keras.utils.set_random_seed(42)
    # Init = keras.initializers.GlorotUniform(seed=42)
    tf.keras.utils.set_random_seed(0)
    Init = keras.initializers.RandomNormal(seed=0)

    input_1 = keras.layers.Input(shape=(input_dim1,), name="numerical_input")
    input_2 = keras.layers.Input(shape=(1,), name="group_input")

    emb = keras.layers.Embedding(input_dim=num_groups,
                                 output_dim=embedding_dim,
                                 embeddings_initializer=Init,
                                 name="state_embedding")(input_2)
    emb_flat = keras.layers.Flatten(name="embedding_flat")(emb)

    bn_in = keras.layers.BatchNormalization(name="bn_input")(input_1)
    h = keras.layers.Concatenate(name="fusion_concat")([bn_in, emb_flat])

    h = keras.layers.Dense(64, kernel_initializer=Init, name="dense_1")(h)
    h = keras.layers.BatchNormalization(name="bn_1")(h)
    h = keras.layers.Activation("relu", name="act_1")(h)
    h = keras.layers.Dropout(0.2, name="drop_1")(h)

    h = keras.layers.Dense(32, kernel_initializer=Init, name="dense_2")(h)
    h = keras.layers.BatchNormalization(name="bn_2")(h)
    h = keras.layers.Activation("relu", name="act_2")(h)
    h = keras.layers.Dropout(0.1, name="drop_2")(h)

    h = keras.layers.Dense(16, kernel_initializer=Init, name="dense_3")(h)
    h = keras.layers.Activation("relu", name="act_3")(h)
    # h = keras.layers.Dropout(0.2, name="drop_3")(h)

    # h = keras.layers.Dense(16, kernel_initializer=Init, name="dense_4")(h)
    # h = keras.layers.Activation("relu", name="act_4")(h)
    # # h = keras.layers.Dropout(0.2, name="drop_3")(h)

    out = keras.layers.Dense(1, activation='softplus', name='output')(h)

    m = keras.Model(inputs=[input_1, input_2], outputs=out)

    lr_schedule = keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=0.005,
        decay_steps=35 * (len(X) * 4 // 5 // 128)
    )
    m.compile(
            #   loss="mean_squared_error",
              loss = tf.keras.losses.Huber(delta=5.0),
              optimizer=keras.optimizers.Adam(learning_rate=lr_schedule),
              metrics=["mae"]
              )
    return m

In [ ]:
# cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=5)
# mse_improved = []

# num_features = [
#     'log_pcterr', 'pcterr_zero', 'inv_pcterr',
#     'log_Uppm', 'log_adjwt',
#     'x', 'y',
#     'floor', 'is_basement',
#     'floor_x_log_Uppm', 'x_x_log_Uppm', 'y_x_log_Uppm',
#     'rep', 'wave', 'dupflag'
# ]
# cat_features = ['room', 'typebldg']
# grp_col = ['grp_id']

# print("=== 5-Fold Stratified CV on Improved Neural Network Model ===")
# for fold, (train_idx, val_idx) in enumerate(cv.split(X, X['state2']), start=1):
#     X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
#     X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

#     fe = RadonFeatureEngineering()
#     X_train_fe = fe.fit_transform(X_train)
#     X_val_fe = fe.transform(X_val)

#     scaler = StandardScaler()
#     X_tr_num = pd.DataFrame(scaler.fit_transform(X_train_fe[num_features]), columns=num_features, index=X_train.index)
#     X_va_num = pd.DataFrame(scaler.transform(X_val_fe[num_features]), columns=num_features, index=X_val.index)

#     ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
#     X_tr_cat = pd.DataFrame(ohe.fit_transform(X_train_fe[cat_features]), index=X_train.index)
#     X_va_cat = pd.DataFrame(ohe.transform(X_val_fe[cat_features]), index=X_val.index)

#     oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
#     X_tr_grp = pd.DataFrame(oe.fit_transform(X_train_fe[grp_col]), index=X_train.index)
#     X_va_grp = pd.DataFrame(oe.transform(X_val_fe[grp_col]), index=X_val.index)

#     X_tr_all = pd.concat([X_tr_num, X_tr_cat], axis=1)
#     X_va_all = pd.concat([X_va_num, X_va_cat], axis=1)

#     num_groups = len(X['state2'].unique())
#     model = build_improved_NN_model(input_dim1=X_tr_all.shape[1], num_groups=num_groups)

#     model.fit([X_tr_all, X_tr_grp],
#               np.log1p(y_train),
#               epochs=35, batch_size=128, verbose=0)

#     y_pred_log = model.predict([X_va_all, X_va_grp], verbose=0)
#     y_pred = np.expm1(y_pred_log).ravel()

#     mse_ = mean_squared_error(y_val, y_pred)
#     mse_improved.append(mse_)
#     print(f"Fold {fold} : MSE = {mse_:.6f}")

# print(f"Mean Improved CV MSE: {np.mean(mse_improved):.6f},{np.std(mse_improved):.6f}")


In [ ]:
num_features = [
    'log_pcterr', 'pcterr_zero', 'inv_pcterr',
    'log_Uppm', 'log_adjwt',
    'x', 'y',
    'floor', 'is_basement',
    'floor_x_log_Uppm', 'x_x_log_Uppm', 'y_x_log_Uppm',
    'rep', 'wave', 'dupflag'
]
cat_features = ['room', 'typebldg']
grp_col = ['grp_id']

fe = RadonFeatureEngineering()
X_fe = fe.fit_transform(X)
df_test_fe = fe.transform(df_test)

scaler = StandardScaler()
X_tr_num = pd.DataFrame(scaler.fit_transform(X_fe[num_features]), columns=num_features, index=X.index)
df_test_num = pd.DataFrame(scaler.transform(df_test_fe[num_features]), columns=num_features, index=df_test.index)

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_tr_cat = pd.DataFrame(ohe.fit_transform(X_fe[cat_features]), index=X.index)
df_test_cat = pd.DataFrame(ohe.transform(df_test_fe[cat_features]), index=df_test.index)

oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_tr_grp = pd.DataFrame(oe.fit_transform(X_fe[grp_col]), index=X.index)
df_test_grp = pd.DataFrame(oe.transform(df_test_fe[grp_col]), index=df_test.index)

X_tr_all = pd.concat([X_tr_num, X_tr_cat], axis=1)
df_test_all = pd.concat([df_test_num, df_test_cat], axis=1)

num_groups = len(X['state2'].unique())
model = build_improved_NN_model(input_dim1=X_tr_all.shape[1], num_groups=num_groups)

model.fit([X_tr_all, X_tr_grp],
          np.log1p(y),
          epochs=35, batch_size=128, verbose=0)

y_pred_log = model.predict([df_test_all, df_test_grp], verbose=0)
y_pred = np.expm1(y_pred_log).ravel()

In [ ]:
pY = pd.DataFrame(y_pred, index=np.arange(len(df_test))+1, columns=['y'])